In [ ]:
%run Latex_macros.ipynb

# Reinforcement Learning: the whole story in one pass

## Where this session fits

- **Series goal** (reinforcement learning, RL)
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position**
    - this is the first session: a tour of the whole series
    - the sessions after it are deep dives into each part
- **This session's goal**
    - build one picture of RL that every later session fills in
- **By the end you can**
    - describe the RL loop: agent, environment, state, action, reward
    - explain why the *return*, not a single reward, is what the agent maximizes
    - name what makes RL hard
    - tell apart three things an agent can learn: a model, a value, a policy
    - say where language models fit


# 1. Learning by trial and error

## Example: teaching Microduck to walk

- [Meet Microduck, the tiny robot you can teach new tricks](https://www.youtube.com/watch?v=reiTh7K4KSc) (Pollen Robotics)
- [4096 Microducks learn to walk: Reinforcement Learning, explained](https://www.youtube.com/watch?v=N3evPHMl0aI) (Rémi Fabre)
- Microduck is a small walking robot; the policy controls its $14$ motorized joints
- everything we show is in simulation: the pictures and videos are from the simulator, not the physical robot
- it is only told how well it did (a reward), never which joint angles to use


**The loop**

- the **agent** is the learner: here, a neural network
    - it is trained with PPO (proximal policy optimization), introduced in section 7
- it senses the **state** (what the robot measures about itself; strictly an *observation*, since it never sees the whole world)
- its **policy** maps the state to an **action** (motor targets)
- the **environment** responds
    - with a **reward**: a number saying how good that was
    - and a new state
- then the loop repeats

<img src="images/microduck_loop.png" width=70%>


**What is Microduck's reward?** (simplified)

- it is a sum of a few simple terms, added up at every step
    - **go where it is told**: follow a commanded walking speed and direction
    - **stay upright**: reward for keeping the trunk level
    - **walk, don't slide**: reward for lifting the feet off the ground in turn
    - **move smoothly**: a penalty for jerky joint motion
- falling ends the episode, so the future rewards are lost
- the designers choose these terms; the network is never told *how* to walk
- the actual terms and weights are more detailed (see the Microduck training code)


**A smaller world: Frozen Lake**

- walk from the start **S** (the elf) to the goal **G** (the gift) across frozen squares **F**, avoiding holes **H** (the blue pools)
- the $16$ squares are numbered $0$ to $15$, row by row from the top left (the later pictures show the numbers)
- state: which square we are on; action: left, down, right, up
- reward: $1$ on reaching **G**, $0$ otherwise; the episode ends at **G** or at a hole

<img src="images/Frozen_Lake_env.png" width=30%>

- the real game is slippery: a move goes the intended way only one time in three, otherwise it slides to one of the two sides
- our first pictures use a non-slippery version; a later session returns to slipping


**An episode, in symbols**

- (**S**, **G**, **F**, **H** label squares; they are not the symbols below)
- one **episode** is a sequence
$$
\stateseq_0, \actseq_0, \rewseq_1, \stateseq_1, \actseq_1, \rewseq_2, \ldots
$$
- the reward that follows action $\actseq_\tt$ is written $\rewseq_{\tt+1}$
- capital-style symbols such as $\stateseq_\tt$ are random quantities at step $\tt$; a plain $\state$ or $\act$ is one particular value


# 2. Reward is feedback on quality, not a target to copy

- **supervised learning** has a target
    - the loss is smallest when the answer matches the target exactly
    - it is imitation
- **reinforcement learning** has a reward
    - it says how good an outcome was, not what the right answer is
    - the learner must discover what to do


**Example: a model answers "How are you today?"**

| Answer | Supervised loss (target: "I feel fine, thank you") | Reward (a judge, a person or a program, scores quality) |
|:--|:--|:--|
| I feel fine, thank you | zero | high |
| Great, thanks for asking | large | high |
| The sky is blue | large | low |

- a loss that compares with one exact target cannot tell the second answer from the third
- a reward can


# 3. The return: why the whole trajectory matters

- an action can pay off many steps later (which action earned the reward? this is the **credit assignment** problem)
    - on Frozen Lake the only reward comes at the very end
- so the agent maximizes the **return**: the rewards from step $\tt$ onward
    - the return is random; the goal is to maximize its average over episodes, the **expected return**
$$
\rlret_\tt = \sum_{k=0}^{|\tau|-\tt-1} \disc^k \, \rewseq_{\tt+k+1}
$$
- $\tau$ is the episode (also called a trajectory), and $|\tau|$ is its number of steps
- $\disc$ is the **discount factor**, between $0$ and $1$
    - smaller $\disc$ means the distant future counts less
    - $\disc = 1$ means no discounting


**Example: Frozen Lake**

- suppose an episode reaches the goal after $n$ steps
    - the only nonzero reward is $\rewseq_n = 1$
    - so the return from the start is $\rlret_0 = \disc^{n-1}$
- with $\disc = 0.9$ and $n = 6$, $\rlret_0 = 0.9^5 \approx 0.59$
- shorter routes earn a larger return (when $\disc < 1$), so the agent prefers them

| route length $n$ | return $\rlret_0 = 0.9^{n-1}$ |
|:--:|:--:|
| $6$ (the shortest) | $0.59$ |
| $8$ | $0.48$ |
| $10$ | $0.39$ |


# 4. Why it is hard

- **sparse reward**
    - Frozen Lake pays only at the goal, so most early episodes teach nothing
    - RL strengthens what the agent already does some of the time: exploration finds it, and RL **amplifies** it; it cannot create what is never tried
    - a robot's small rewards give partial credit along the way
    - for a language model, SFT first makes the right behavior possible
        - SFT = supervised fine-tuning on example answers
    - Microduck's designers add many small rewards: stay upright, walk with alternating feet, move smoothly
- **exploration versus exploitation**
    - exploit: repeat what has worked so far
    - explore: try something else that might work better
    - behavior never tried can never be rewarded, so exploring decides what can be learned
- **random outcomes**
    - on slippery ice the same action leads to different squares
- **partial information**
    - the duck gets an **observation** (gyroscope, gravity direction, joint angles and speeds, its last action, and its commands), not the whole world
    - we call this its state; strictly it is a partial view of the world's state


**Unexpected strategies, and reward hacking**

- the agent maximizes the reward **as written**, not as intended, so it can find strategies nobody specified
    - when such a strategy exploits a gap between the reward and what the designers wanted, it is called **reward hacking**
- an example from the simulator: Microduck learned to climb an alternating-tread ladder onto a desk
    - in a way that a person would not have specified
    - it braces one foot between two treads, then steps up with the other foot
    - [video of the climb](https://github.com/Vottivott/microduck-playground/blob/main/experiments/desk-climb/media/preview.mp4)
    - a person would instead place a foot on the next tread and push up from the knee

<table>
    <center><strong>Foot braced between two steps (post by Hannes von Essen)</strong></center>
    <tr><td><img src="images/microduck_stairs_reward_hacking.png" width=40%></td></tr>
</table>


# 5. Three things an agent could learn

- a **model**: what the world does in response
    - here "model" means a model of the environment, not a language model
    - $\transp(\state', \rew \mid \state, \act)$: the chance of the next state and reward, given a state and action
- a **value**: how good it is to be here (when following a policy $\pi$, defined next)
    - the expected return from state $\state$ when following policy $\pi$
    - $\statevalfun_\pi(\state) = \Exp{\pi}\left[ \rlret_\tt \mid \stateseq_\tt = \state \right]$
    - the **action value** $\actvalfun_\pi(\state, \act)$ is the same, but starting with action $\act$
    - under optimal play, the value of a state is that of its best action
- a **policy**: what to do here
    - $\pi(\act \mid \state)$: the chance of each action, given the state
- different methods learn different ones; later sessions take them in turn


<img src="images/frozenlake_three_things.png" width=95%>

- Frozen Lake: the model panel is slippery ice; the value and policy panels are non-slippery with $\disc = 0.9$
- the value $0.59$ at the start equals the return of the 6-step route in section 3
    - without slipping, every episode of the best policy gives the same return
- the value and policy panels show optimal play ($\statevalfun_*$); the policy simply picks the best action in each square


# 6. What learning a value table looks like

- start with every value at zero
- the agent plays episodes and updates its estimates
- information spreads backward from the goal, a little with each episode
- Frozen Lake (non-slippery), learned by Q-learning (a later session)
    - each square shows the best **action value** $\actvalfun(\state, \act)$ found so far (the "Q" in Q-learning), and the arrow is that action
    - snapshots after $0$, $70$, $145$ and $19{,}999$ episodes
    - each number is the best action value in that square (like a state value)
    - once the agent settles on a route it rarely visits other squares (here $2$ and $3$), so they stay near $0$
    - this run is not fully converged
        - squares $1$ and $10$ do not point to the best action
        - square $6$ reads $0.59$, the optimum is $0.81$
    - the color scale differs between pictures
    - the optimal values are in the previous section's figure

<table>
    <tr>
        <td><center>episode 0</center><img src="images/qlearning_policy_evolution_0.png"></td>
        <td><center>episode 70</center><img src="images/qlearning_policy_evolution_70.png"></td>
        <td><center>episode 145</center><img src="images/qlearning_policy_evolution_145.png"></td>
        <td><center>episode 19,999</center><img src="images/qlearning_policy_evolution_final.png"></td>
    </tr>
</table>


**Tables stop working**

- Frozen Lake has $16$ squares: a table has one entry per square and action
- Microduck's state is $61$ continuous numbers: no table can hold it
- so we learn a *function*, a neural network, instead
    - for a value, or directly for the policy

<img src="images/table_vs_network.png" width=80%>


# 7. Learning a policy directly

- the policy is a function with parameters $\theta$: $\pi_\theta(\act \mid \state)$
- the idea
    - after an episode, look at the return that followed each action
    - make actions that were followed by a high return **more likely**
    - make actions that were followed by a low return **less likely**
        - in practice the return is replaced by the **advantage** $\advseq_\tt$: how much better than a typical return (a later session)
- in one line, with the plain return and ignoring discounting for now ($\disc = 1$; the exact form has an extra factor $\disc^\tt$)
$$
\theta \leftarrow \theta + \alpha \, \nabla_\theta \log \pi_\theta(\actseq_\tt \mid \stateseq_\tt) \, \rlret_\tt
$$
- $\alpha$ is the learning rate (step size)
- $\nabla_\theta \log \pi_\theta(\actseq_\tt \mid \stateseq_\tt)$ points in the direction that makes that action more likely; the return sets how hard to push
- a later session derives the exact form and the methods built on it


**Microduck's policy is an instance of this**

- a neural network $\pi_\theta$, whose output is a distribution over motor targets (training samples from it)
    - observations in, motor targets out (the sizes are in the loop figure)
- trained with PPO (proximal policy optimization), a refinement of this idea
    - thousands of simulated ducks learn in parallel
- the simulator is only a stand-in for the world: PPO never looks inside it
- everything here is in simulation


# 8. Where language models fit

- a language model is a policy $\pi_\theta(\act \mid \state)$
    - state: the prompt $\x$ and the response so far, $\stateseq_\tt = \y_{[0:\tt-1]}$
    - action: the next token (a token is a word or a piece of a word)
    - episode: one complete response
    - reward: a judge's score, or a score learned from human preferences (a reward model; a later session)
    - as on Frozen Lake, the reward often arrives only at the end of the episode
- the same family of methods (PPO and its relatives) is used to train both Microduck and assistants
- this course builds the foundations
- a later course, on reinforcement fine-tuning of language models, builds on them


# What we established, and what it sets up

- **Established**
    - the loop: agent, environment, state, action, reward
    - reward is feedback on quality, not a target to copy
    - the return, $\rlret_\tt$, is what the agent maximizes
    - what makes it hard: sparse reward, exploration, reward hacking, randomness, partial information
    - three things to learn: a model, a value, a policy
    - language models are policies too
- **What it sets up**
    - values, the Bellman equation and planning with a model
    - learning from experience: Monte Carlo, temporal difference, Q-learning
    - on-policy versus off-policy, and deep Q-networks
    - policy gradients: the idea, the theorem, advantage and PPO
    - reinforcement learning for language models and preferences
